# College Admission OpenEnv — TRL Training (Kaggle T4)

This notebook runs `train_trl_kaggle.py` end-to-end:
- Connects to the OpenEnv HTTP environment
- Trains a small model with TRL + QLoRA
- Compares trained vs random/untrained baselines
- Saves labeled plots and metrics
- Optional HF Hub / Space push


In [ ]:
!pip -q install "transformers>=4.46.0" "trl>=0.11.0" "peft>=0.13.0" \
                "accelerate>=0.34.0" "bitsandbytes>=0.43.0" "datasets>=2.20.0" \
                "huggingface_hub>=0.25.0" "pandas>=2.1.0" "matplotlib>=3.8.0" \
                "requests>=2.31.0

: 

In [ ]:
import os
import subprocess
import sys

# --- Required runtime config ---
os.environ["COLLEGE_ENV_BASE_URL"] = "https://Knight09-college-admission-env.hf.space"
os.environ["BASE_MODEL_NAME"] = "Qwen/Qwen2.5-0.5B-Instruct"
os.environ["OUTPUT_DIR"] = "./trl_runs/college_env_qwen25_05b"

# --- Training/eval knobs (T4-friendly defaults) ---
os.environ["TRAIN_EPISODES_PER_TEMPLATE"] = "110"
os.environ["MAX_TRAIN_STEPS"] = "600"
os.environ["TRAIN_BATCH_SIZE"] = "8"
os.environ["GRAD_ACCUM_STEPS"] = "4"
os.environ["EVAL_EPISODES_PER_TASK"] = "12"

# --- Optional HF Hub push ---
# os.environ["PUSH_TO_HUB"] = "1"
# os.environ["HF_TOKEN"] = "<your_hf_token>"
# os.environ["HUB_MODEL_REPO"] = "<username>/<model-repo-name>"

# --- Optional HF Space dashboard push ---
# os.environ["PUSH_TO_SPACE_DASHBOARD"] = "1"
# os.environ["HF_SPACE_REPO"] = "<username>/<space-repo-name>"

cmd = [sys.executable, "train_trl_kaggle.py"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

output_dir = Path(os.environ.get("OUTPUT_DIR", "./trl_runs/college_env_qwen25_05b"))
artifact_dir = output_dir / "artifacts"

summary_path = artifact_dir / "evaluation_summary.csv"
episodes_path = artifact_dir / "evaluation_episodes.csv"
print("Summary:", summary_path)
print("Episodes:", episodes_path)
print("\nEvaluation summary table:")
display(pd.read_csv(summary_path).head(30))

for img_name in ["training_loss_curve.png", "episode_return_curve.png", "task_score_bar.png"]:
    img_path = artifact_dir / img_name
    print(f"\n{img_name} -> {img_path}")
    display(Image(filename=str(img_path)))
